In [1]:
import os, glob
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
pd.set_option('display.max_columns', None)

from scipy.fft import fft
from sklearn.model_selection import LeaveOneOut
import xgboost as xgb
from sklearn.metrics import recall_score, precision_score, balanced_accuracy_score, roc_auc_score

In [2]:
DATA_PATH   = "/home/justine/code/Maelle05/DyslexIA/Dataset"
LABELS_PATH = "/home/justine/code/Maelle05/DyslexIA/Dataset/dyslexia_class_label.csv"

TARGET_LEN = 2000 # A FINE TUNER
GAZE_COLS  = ['x_left', 'y_left', 'x_right', 'y_right']

files_raw = [f for f in glob.glob(os.path.join(DATA_PATH, "*clean.csv")) if not f.endswith("app_clean.csv")]
labels = pd.read_csv(LABELS_PATH)

In [3]:
all_lengths = []

for f in files_raw:
    df = pd.read_csv(f)
    all_lengths.append(len(df))

print("min:", min(all_lengths))
print("max:", max(all_lengths))
print("mean:", sum(all_lengths)/len(all_lengths))

min: 992
max: 97271
mean: 8768.2


In [4]:
def to_fixed_length(signal, target_len):
    """Interpolation linéaire vers une longueur fixe."""
    x_old = np.linspace(0, 1, len(signal))
    x_new = np.linspace(0, 1, target_len)
    return interp1d(x_old, signal, kind='linear')(x_new)

In [5]:
def extract_features(df, gaze_cols, target_len):
    signals = {}
    for col in gaze_cols:
        signals[col] = to_fixed_length(df[col].values.astype(float), target_len)

    # Divergence binoculaire moyenne — écart horizontal gauche/droite
    mean_binocular_divergence = np.mean(np.abs(signals['x_left'] - signals['x_right']))

    # Percentile 90 de la vitesse verticale — y_left
    vel_yl = np.abs(np.diff(signals['y_left']))
    p90_vertical_velocity_left = np.percentile(vel_yl, 90)

    # Taux de saccades horizontales — x_left
    vel_xl = np.abs(np.diff(signals['x_left']))
    saccade_rate_x_left = np.sum(vel_xl > np.mean(vel_xl) + 2*np.std(vel_xl)) / len(vel_xl)

    # Énergie normalisée de la bande la plus haute fréquence (bande 10/10) — x_left
    spectrum = np.abs(fft(signals['x_left']))[:len(signals['x_left'])//2]
    bands = np.array_split(spectrum, 10)
    energies = np.array([np.sum(b**2) for b in bands])
    high_freq_band_energy_x_left = (energies / (np.sum(energies) + 1e-8))[9]

    return np.array([mean_binocular_divergence, p90_vertical_velocity_left, saccade_rate_x_left, high_freq_band_energy_x_left])

In [6]:
rows = []
for f in files_raw:
    df = pd.read_csv(f)
    sid = os.path.basename(f).split('_')[1]
    label_row = labels[labels['subject_id'] == sid]
    rows.append({'sid': sid,
                 'label': label_row['class_id'].values[0],
                 'features': extract_features(df, GAZE_COLS, TARGET_LEN)})

sids = np.array([r['sid']      for r in rows])
X    = np.array([r['features'] for r in rows])
y    = np.array([r['label']    for r in rows])
print(f"X : {X.shape} | Classes : {np.bincount(y)} (0=non-dys, 1=dys)")

X : (255, 4) | Classes : [123 132] (0=non-dys, 1=dys)


In [7]:
OPTIMAL_THRESHOLD = 0.35

loo = LeaveOneOut()
y_true, y_pred, y_scores = [], [], []

for train_idx, test_idx in loo.split(X, y):

    clf = xgb.XGBClassifier(
        max_depth        = 2,
        n_estimators     = 50,
        learning_rate    = 0.1,
        reg_lambda       = 2,
        colsample_bytree = 0.8,
        random_state=42, verbosity=0
    )

    clf.fit(X[train_idx], y[train_idx])

    y_true.append(y[test_idx][0])
    y_scores.append(clf.predict_proba(X[test_idx])[0, 1])
    y_pred.append(int(y_scores[-1] >= OPTIMAL_THRESHOLD))

y_true, y_pred, y_scores = map(np.array, [y_true, y_pred, y_scores])
print(f'  Recall            : {recall_score(y_true, y_pred):.4f}')
print(f'  Precision         : {precision_score(y_true, y_pred):.4f}')
print(f'  Balanced Accuracy : {balanced_accuracy_score(y_true, y_pred):.4f}')
print(f'  AUC-ROC           : {roc_auc_score(y_true, y_scores):.4f}')

  Recall            : 0.8712
  Precision         : 0.7877
  Balanced Accuracy : 0.8096
  AUC-ROC           : 0.9068


In [8]:
final_clf = xgb.XGBClassifier(
    max_depth        = 2,
    n_estimators     = 50,
    learning_rate    = 0.1,
    reg_lambda       = 2,
    colsample_bytree = 0.8,
    random_state=42, verbosity=0
)

final_clf.fit(X, y)

final_clf.save_model('dyslexia_xgb.json')